In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

In [2]:
from langchain_ollama import ChatOllama

from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_core.chat_history import InMemoryChatMessageHistory

from app.prompts import RAG_PROMPT
from app.retriever import get_retriever
from app.vectordb import load_vector_db

c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
vector_db = load_vector_db()

retriever = get_retriever(vector_db)

llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [4]:
store = {}

In [5]:
def get_session_history(session_id: str):

    if session_id not in store:

        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]

In [6]:
def chat(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(

        doc.page_content

        for doc in docs

    )

    prompt = RAG_PROMPT.invoke(

        {

            "context":context,

            "question":question

        }

    )

    return llm.invoke(prompt)

In [7]:
response = chat("What is diabetes?")

print(response.content)

 Diabetes Mellitus is a condition characterized by high levels of glucose (sugar) in the blood due to problems with insulin production or function.


In [9]:
from langchain_core.runnables import RunnableLambda

In [10]:
chat_chain = RunnableLambda(chat)

In [11]:
chat_with_memory = RunnableWithMessageHistory(

    chat_chain,

    get_session_history,

    input_messages_key="question",

    history_messages_key="history"

)

In [12]:
history = get_session_history("user1")

print(history.messages)

[]


In [13]:
history1 = get_session_history("alice")

history2 = get_session_history("bob")

In [14]:
history = get_session_history(

    "session1"

)

history.add_user_message(

    "Hello"

)

history.add_ai_message(

    "Hi"

)

print(history.messages)

[HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
